# Module 11 — KServe (churn + clothing model): commented walkthrough

Alexey's lecture code (lessons 11.1 – 11.7), with comments that explain **three things** for every step:

- `# GOAL`   what this step is for, in plain words
- `# WHY`    the machine-learning / deployment logic behind it (what could go wrong if we skipped it)
- `# PYTHON` how the Python syntax works (`# SYNTAX` in YAML and Dockerfiles), so you can write it yourself later

The original, uncommented code is untouched in `../notebooks/code/`.

**How this notebook works.** Module 11 has no lecture notebook: it is scripts and YAML. The Python parts
(training the churn pipeline, building requests, the transformer's pre/post-processing) run here on your laptop.
The Kubernetes parts are shown with `!cat` from commented copies in `commented/`, and the `kubectl` commands are in
markdown cells: run them in a terminal against the kind cluster from Module 10. The request cells catch
`ConnectionError`, so the notebook runs end to end without a cluster and works for real once the port-forward is up.

**Not executed when this was built:** anything needing Kubernetes (KServe install, `kubectl apply`, the requests to
the cluster, EKS), Docker builds, TensorFlow and `keras_image_helper`. The rest of the notebook was executed.
Separately, `commented/churn-train-2026.py` was trained with scikit-learn 1.5.2 and served with KServe 0.21's
sklearn server run locally (no Kubernetes): it returned probabilities for the dict-of-lists request below.

Differences from the lecture:
- the data is read from the local `data-week-3.csv` (same Telco file as Modules 3-4), not from the GitHub URL;
- text columns are found with `select_dtypes(include=['object', 'string'])`. On pandas 3 the lecture's
  `df.dtypes == 'object'` selects nothing, every churn label becomes 0, and scikit-learn 1.9 stops with a
  misleading *"'liblinear' solver does not support multiclass classification"*;
- the model is saved as `model/model.joblib` (one model file per folder, as the KServe server requires);
- the repo's `model.joblib` was pickled with scikit-learn **1.0**. On 1.9 it unpickles with
  `InconsistentVersionWarning`s, predicts, but crashes as soon as you print it
  (*"'Pipeline' object has no attribute 'transform_input'"*). A cell below shows it. Version mismatches in the
  server fail harder: see 11.4;
- **outdated 2022 versions**, flagged where they appear: kind 0.11.1 / kubectl 1.21 / `kindest/node:v1.21.1`,
  KServe 0.8 `quick_install.sh` (gone from the repo), Istio 1.6.2, EKS 1.21, Python 3.8 images,
  `kserve==0.7.0` with `KFModel`/`KFServer`, scikit-learn 1.0. Current KServe (0.20 / 0.21) needs Kubernetes >= 1.32;
- `apiVersion: serving.kserve.io/v1beta1` is **not** outdated: it is still the InferenceService version today.
  `predictor.sklearn:` is still accepted; current docs write `predictor.model.modelFormat.name: sklearn`;
- **current KServe turns dict instances into a pandas DataFrame** before calling the model. The lecture's
  request and its `DictVectorizer` pipeline both fail on that. The 2026 variant (`OneHotEncoder` in a
  `ColumnTransformer`, instances as `{column: [value]}`, `PREDICT_PROBA=true`) is built next to the lecture version;
- the custom predict_proba image of lesson 11.4 (`agrigorev/sklearnserver:3.8-1.0-predict-proba`) is replaced by
  the stock server's `PREDICT_PROBA` environment variable;
- the image transformer's `Dockerfile` copies `image_tranformer.py` (typo) instead of `image_transformer.py`;
  fixed in `commented/Dockerfile`. The folder is spelled `image_transfomer` in the repo; kept as is;
- `storageUri: http://172.31.13.90:8000/...` is a private IP of Alexey's machine; replace it with your own location.

## 11.1 Overview

**KServe** (formerly KFServing, part of Kubeflow) is a Kubernetes add-on for serving models. In Module 10 you wrote
a Deployment and a Service for TF Serving and another pair for the gateway, by hand. KServe adds one resource,
the **InferenceService**: you say *which model, in which format, stored where*, and it creates the deployments,
services, routing and autoscaling (including scale-to-zero in Knative mode).

The same **two-tier architecture** as Module 10:

| Tier | Module 10 | Module 11 |
|---|---|---|
| pre/post-processing | `gateway.py` (Flask) Deployment + Service | `transformer:` in the InferenceService |
| model | TF Serving Deployment + Service | `predictor:` (sklearn, tensorflow, ... runtimes) |

We deploy two models: the **churn** pipeline (scikit-learn, 11.3-11.4) and the **clothing** classifier
(TensorFlow + transformer, 11.5-11.6). Simulation analogy: the InferenceService is a *parametrised library object*.
You set its parameters and the platform instantiates the internals, instead of modelling each station yourself.

## 11.2 Running KServe locally

**Lecture (2022), outdated.** kind 0.11.1, kubectl 1.21.1, `kind create cluster --image kindest/node:v1.21.1`,
then `curl -s https://raw.githubusercontent.com/kserve/kserve/release-0.8/hack/quick_install.sh | bash`.
That script is gone from current releases, and current KServe does not support Kubernetes 1.21.

**2026 equivalent** (run in a terminal; not tested here). KServe 0.20+ needs a Kubernetes **server >= v1.32**:

```bash
brew install kind kubectl helm          # macOS; recent kind versions start a recent Kubernetes
kind create cluster --name kserve
kubectl version                         # "Server Version" must be >= v1.32. Module 10's kindest/node:v1.23.0 is too old

# Knative mode = the lecture's setup (Istio ingress, Host-header routing, scale to zero).
# Take the script from the LATEST release page; the KServe docs showed v0.20.0 in Oct 2026.
curl -sL "https://github.com/kserve/kserve/releases/download/v0.20.0/kserve-knative-mode-full-install-with-manifests.sh" | bash

kubectl get pods -A                     # wait until everything is Running
```

In [ ]:
# GOAL: the "hello world" InferenceService: a ready-made iris model from KServe's public bucket.
# PYTHON: '!' runs a shell command from the notebook; cat prints the file.
!cat commented/iris-example.yaml

Deploy it and open a tunnel to the ingress gateway (two terminals):

```bash
kubectl apply -f commented/iris-example.yaml
kubectl get inferenceservice sklearn-iris          # wait for READY=True
kubectl port-forward -n istio-system service/istio-ingressgateway 8080:80
```

If the service name differs in your install, `kubectl get svc -n istio-system` lists it.

In [ ]:
# GOAL: iris-test.py: send two flowers to the model through the gateway.
# PYTHON: requests is the HTTP client from Module 5.
import requests

service_name = 'sklearn-iris'

# WHY: one gateway serves EVERY InferenceService. It routes by the Host header <name>.<namespace>.<domain>;
#      example.com is the default domain of a local install. Without the header the gateway returns 404.
host = f'{service_name}.default.example.com'

actual_domain = 'http://localhost:8080'
# WHY: /v1/models/<name>:predict is KServe's V1 protocol, the same URL shape as TF Serving's REST API.
url = f'{actual_domain}/v1/models/{service_name}:predict'

headers = {'Host': host}

request = {
    "instances": [
        [6.8,  2.8,  4.8,  1.4],
        [6.0,  3.4,  4.5,  1.6]
    ]
}

# PYTHON: try/except catches the error when no port-forward is running, so the notebook keeps going.
try:
    response = requests.post(url, json=request, headers=headers, timeout=10)
    print(response.json())       # expected: {'predictions': [1, 1]}
except requests.exceptions.ConnectionError:
    print('No cluster on localhost:8080: start the port-forward first.')

## 11.3 Deploying a Scikit-Learn model with KServe

`churn-train.py`: a small churn model (3 features) packed **with its DictVectorizer** into one `Pipeline`,
saved with joblib. The sklearn server will load that single file and call `.predict()` on the request rows.

In [ ]:
# PYTHON: 'import X as Y' loads a library under a short nickname.
import os
import warnings

import joblib                  # saves / loads fitted scikit-learn objects (one file = the whole pipeline)
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

import sklearn
# WHY: the version that pickles the model must match the version that unpickles it in the server.
#      Note what you have here (probably 1.9) against what the server image ships (KServe 0.21: 1.5.2).
sklearn.__version__, pd.__version__

In [ ]:
# GOAL: load and clean the Telco data exactly as in Module 3.
df = pd.read_csv('data-week-3.csv')

df.columns = df.columns.str.lower().str.replace(' ', '_')

# WHY: select_dtypes works on old pandas (text = 'object') AND pandas 3 (text = 'str').
#      The lecture's df.dtypes == 'object' finds no columns on pandas 3 -> nothing gets cleaned below.
categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn == 'yes').astype(int)

# Sanity check: ~0.27. If it prints 0.0 the cleaning did not run and training will fail.
df.churn.mean()

In [ ]:
# GOAL: only three features, so the JSON request stays readable.
# WHY: 11.3 is about deployment, not model quality. (No train/validation split either: same reason.)
numerical = ['tenure', 'monthlycharges']
categorical = ['contract']

# PYTHON: to_dict(orient='records') -> a list with one {column: value} dict per customer.
dicts = df[categorical + numerical].to_dict(orient='records')
dicts[:2]

In [ ]:
# GOAL: vectorizer + model in ONE object.
# WHY: the server loads exactly one file. If the DictVectorizer were saved separately, nothing in the cluster
#      could one-hot encode the incoming request. "Preprocessing travels with the model."
# PYTHON: Pipeline([(name, step), ...]); fit runs fit_transform on each step, then fit on the last one.
pipeline = Pipeline([
    ('vectorizer', DictVectorizer(sparse=False)),
    ('model', LogisticRegression(C=1.0, solver='liblinear'))
])

pipeline.fit(dicts, df.churn.values)

In [ ]:
# GOAL: the two customers of churn-test.py, scored locally: this is what the service should answer.
customers = [
    {'contract': 'one_year', 'tenure': 34, 'monthlycharges': 56.95},
    {'contract': 'month-to-month', 'tenure': 13, 'monthlycharges': 49.95},
]

# WHY: the stock sklearn server calls .predict() -> labels. Both are 0 at the 0.5 threshold, which hides the
#      fact that the second customer is 3-4x riskier. That is the motivation for 11.4 (predict_proba).
# PYTHON: [:, 1] keeps column 1 of predict_proba = P(churn).
pipeline.predict(customers), pipeline.predict_proba(customers)[:, 1].round(3)

In [ ]:
# GOAL: save the artefact the InferenceService will download.
# WHY: the sklearn server accepts a folder with exactly ONE .joblib / .pkl / .pickle file, so give it its own folder.
# PYTHON: exist_ok=True -> no error if the folder already exists.
os.makedirs('model', exist_ok=True)
joblib.dump(pipeline, 'model/model.joblib')

In [ ]:
# GOAL: why you should not reuse the repo's model.joblib: it was pickled by scikit-learn 1.0 in 2022.
# WHY: a pickle stores the object's internal attributes, not a portable format. A newer scikit-learn expects
#      attributes the old object never had. Some methods still work, others crash: "use at your own risk".
#      Same rule in the cluster: train with the scikit-learn version of the server image.
# PYTHON: catch_warnings(record=True) collects the warnings in a list instead of printing them.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    old = joblib.load('../notebooks/code/churn/model.joblib')

print(len(caught), 'warnings, e.g.:', str(caught[0].message)[:90])
print('predict_proba still works:', old.predict_proba(customers)[:, 1].round(3))

try:
    print(old)                                  # printing the pipeline needs an attribute added after 1.0
except AttributeError as e:
    print('AttributeError:', e)

In [ ]:
# GOAL: the InferenceService for the churn model (comments explain every field).
!cat commented/churn-service.yaml

Deploy (terminal). The model file must be reachable from the cluster: the lecture serves it with
`python -m http.server 8000` on an EC2 machine, whose private IP is hard-coded in the YAML. Put yours on S3/GCS
or any URL the cluster can reach, and edit `storageUri`.

```bash
kubectl apply -f commented/churn-service.yaml
kubectl get inferenceservice churn
kubectl logs -l serving.kserve.io/inferenceservice=churn -c storage-initializer   # download problems show here
```

In [ ]:
# GOAL: churn-test.py: the same two customers, through the cluster.
service_name = 'churn'
host = f'{service_name}.default.example.com'
url = f'http://localhost:8080/v1/models/{service_name}:predict'

request = {'instances': customers}

try:
    response = requests.post(url, json=request, headers={'Host': host}, timeout=10)
    print(response.json())
except requests.exceptions.ConnectionError:
    print('No cluster on localhost:8080: start the port-forward first.')

## 11.4 Deploying custom Scikit-Learn images with KServe

The stock server returned labels; we want probabilities. In 2022 the fix was a **custom server image**:
KServe's `sklearn.Dockerfile` with `predict` changed to `predict_proba`, built for Python 3.8 + scikit-learn 1.0,
and tested with plain Docker before going to the cluster (`notebooks/code/README.md`):

```bash
docker run -it --rm -v "$(pwd)/model.joblib:/mnt/models/model.joblib" -p 8081:8080 \
    kserve-sklearnserver:predict_proba-3.8-1.0 --model_dir=/mnt/models --model_name=churn
```

**2026:** no custom image needed. The stock server reads an environment variable: `PREDICT_PROBA=true`.
But the current server also changed **what it hands to your model**, which breaks the lecture pipeline.
The next cells reproduce what the server does, without a cluster.

In [ ]:
# GOAL: mimic how current KServe (checked: 0.21, kserve.utils.get_predict_input) turns "instances" into model input.
# WHY: when the instances are dicts, it builds a pandas DataFrame from EACH dict and stacks them.
# PYTHON: pd.concat([...], axis=0) stacks DataFrames on top of each other.
def server_input(instances):
    return pd.concat([pd.DataFrame(instance) for instance in instances], axis=0)

# The lecture's request: scalar values -> pandas refuses to build a one-row DataFrame from scalars.
try:
    server_input(customers)
except ValueError as e:
    print('ValueError:', e)

In [ ]:
# GOAL: the request shape that DOES become a DataFrame: every value wrapped in a list.
# PYTHON: {k: [v] for k, v in d.items()} is a dict comprehension: same keys, each value inside a one-item list.
customers_2026 = [{k: [v] for k, v in c.items()} for c in customers]

X_server = server_input(customers_2026)
X_server

In [ ]:
# WHY: the lecture pipeline still fails: DictVectorizer wants a list of dicts; iterating a DataFrame yields its
#      column NAMES (strings), hence "'str' object has no attribute 'items'". Same error as the real server.
try:
    pipeline.predict_proba(X_server)
except AttributeError as e:
    print('AttributeError:', e)

In [ ]:
# GOAL: the 2026 variant (= commented/churn-train-2026.py): a pipeline that reads a DataFrame.
# WHY: ColumnTransformer picks columns BY NAME, so it works on whatever DataFrame the server builds.
# PYTHON: remainder='passthrough' keeps the columns we did not list (tenure, monthlycharges) unchanged.
#         handle_unknown='ignore' -> an unseen contract value becomes all zeros instead of an error.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    [('onehot', OneHotEncoder(handle_unknown='ignore'), categorical)],
    remainder='passthrough',
)

pipeline_2026 = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(C=1.0, solver='liblinear')),
])

# PYTHON: fit on a DataFrame (not dicts), so the pipeline remembers the column names.
pipeline_2026.fit(df[categorical + numerical], df.churn.values)

# WHY: same model, same probabilities as the DictVectorizer version, but now fed the server's DataFrame.
pipeline_2026.predict_proba(X_server)[:, 1].round(3), pipeline.predict_proba(customers)[:, 1].round(3)

In [ ]:
# GOAL: the matching 2026 InferenceService: stock runtime, PREDICT_PROBA=true, current modelFormat style.
!cat commented/churn-service-2026.yaml

To deploy the 2026 variant, **do not** upload a model trained in this notebook (scikit-learn 1.9): served by
KServe 0.21's scikit-learn 1.5.2 it fails with *"'LogisticRegression' object has no attribute 'multi_class'"*
(checked). Train it in a matching environment:

```bash
conda create -n kserve-train python=3.12 "scikit-learn=1.5.2" "pandas<3" joblib
conda activate kserve-train
cd practice/commented && python churn-train-2026.py      # -> model_2026/model.joblib
```

Then upload `model_2026/model.joblib`, set `storageUri` and `kubectl apply -f commented/churn-service-2026.yaml`.
Request body: `{"instances": [{"contract": ["one_year"], "tenure": [34], "monthlycharges": [56.95]}, ...]}`.

In [ ]:
# GOAL: churn-test for the 2026 service: wrapped values, answer = [[P(stay), P(churn)], ...].
try:
    response = requests.post(url, json={'instances': customers_2026}, headers={'Host': host}, timeout=10)
    print(response.json())
except requests.exceptions.ConnectionError:
    print('No cluster on localhost:8080: start the port-forward first.')

## 11.5 Serving TensorFlow models with KServe

The clothing classifier from Modules 8-10. `convert.py` turns the Keras `.h5` file into a **SavedModel**, the
format TF Serving reads (the same step as Module 10's lesson 10.2, which also has the download link for
`xception_v4_large_08_0.894.h5`). TF Serving expects a numbered version folder: `clothing-model/1/saved_model.pb`.

In [ ]:
!cat ../notebooks/code/clothes/convert.py

Without a transformer the **client** does the preprocessing (`clothes/test.py`): it downloads the image,
resizes it to 299x299, scales it, and sends the raw tensor as JSON. How big is that request?

In [ ]:
# GOAL: measure the JSON body that test.py sends for ONE image.
# WHY: 299 x 299 x 3 = 268,203 floats written as text. That is why 11.6 moves preprocessing into the cluster:
#      the client then sends a ~40-byte URL, and only the internal hop carries the tensor.
# PYTHON: np.random.rand(...) makes a fake image of the right shape; json.dumps turns it into the request text.
import json

X_fake = np.random.rand(1, 299, 299, 3).astype('float32')
body = json.dumps({'instances': X_fake.tolist()})

print(f'{len(body) / 1e6:.1f} MB of JSON for one image')

## 11.6 KServe transformers

A **transformer** is a KServe model that does not predict: `preprocess` edits the request on the way in,
`postprocess` edits the response on the way out, and KServe calls the predictor in between.
Alexey deploys a prebuilt transformer image configured by environment variables (`clothes-service.yaml`)
and shows how such a transformer is written (`image_transformer.py`).

In [ ]:
!cat commented/clothes-service.yaml

In [ ]:
# NOTE: the kserve 0.7 API (KFModel, KFServer). Renamed in 0.8: see the header of the file.
!cat commented/image_transformer.py

In [ ]:
# GOAL: run the transformer's postprocess logic locally, with a fake predictor response.
# WHY: the logic is plain Python; only the server around it needs kserve. Testing it in isolation is how you
#      would unit-test a transformer before building an image.
classes = [
    'dress', 'hat', 'longsleeve', 'outwear', 'pants',
    'shirt', 'shoes', 'shorts', 'skirt', 't-shirt',
]

def postprocess(response):
    result = []
    for prediction in response['predictions']:
        # PYTHON: zip pairs label i with score i; dict() turns the pairs into {label: score}.
        result.append(dict(zip(classes, prediction)))
    return {'predictions': result}

# Fake TF Serving answer for one image: 10 raw scores (logits), "pants" the highest.
fake_response = {'predictions': [[-1.9, -4.8, -2.1, -4.0, 9.9, -2.9, -3.7, 3.2, -2.6, -4.8]]}

out = postprocess(fake_response)['predictions'][0]
# PYTHON: max(d, key=d.get) returns the key with the largest value.
max(out, key=out.get), out

In [ ]:
# GOAL: the transformer's container. The lecture's COPY line had a typo (image_tranformer.py); fixed here.
!cat commented/Dockerfile

Build and run it on kind (terminal, not tested here). kind cannot pull from your laptop's Docker, so load the image:

```bash
docker build -t image-transformer:v1 -f practice/commented/Dockerfile notebooks/code/image_transfomer
kind load docker-image image-transformer:v1 --name kserve
```

Then reference `image-transformer:v1` under `transformer.containers` instead of `agrigorev/kfserving-keras-transformer`.
With `kserve==0.7.0` pinned the code runs as written; with a current kserve, apply the renames first.
Test with `clothes/test-transformer.py` (the body is just URLs).

## 11.7 Deploying with KServe and EKS

Two pages: `07-kserve-eks.md` holds the **video**; `07-kserve-eks-upd.md` is the **13 July 2022 update** with the
commands to use instead of the video's (no custom domain: call the ELB hostname with a `Host` header).
Both are 2022 and partly outdated:

- `cluster.yaml` asks for Kubernetes **1.21**, which EKS no longer creates. Use a supported version (>= 1.32 for KServe 0.20+);
- the repo's `eks/cluster.yaml` says `eu-west-1`, the update says `us-east-2`: pick one and use it everywhere;
- the update edits Istio to 1.6.2 inside the KServe 0.8 `quick_install.sh`: use the current install script instead;
- `kserve-s3-secret.yaml` (S3 access for `storageUri: s3://...`) is not in the repo: write it from the KServe docs;
- it creates `test_transformer.py` and then runs `test-transformer3.py`: same file, typo;
- **2 x m5.xlarge + ELB cost money by the hour**: `eksctl delete cluster --name mlzoomcamp-eks` when done.

In [ ]:
!cat ../notebooks/code/eks/cluster.yaml

## 11.8 Summary

| Piece | What it does | Watch out (2026) |
|---|---|---|
| InferenceService | one resource = deployment + service + routing + autoscaling | still `serving.kserve.io/v1beta1` |
| sklearn predictor | loads one `.joblib` and calls `predict` | server's scikit-learn version (0.21: 1.5.2); input arrives as a DataFrame |
| `PREDICT_PROBA=true` | probabilities instead of labels | replaces the custom image of 11.4 |
| tensorflow predictor | TF Serving behind the KServe protocol | SavedModel with a version folder |
| transformer | pre/post-processing tier in front of the predictor | `KFModel`/`KFServer` renamed in kserve 0.8 |
| storageUri | where the model is downloaded from | must be reachable from the cluster |

Alexey's verdict (plan.md): less YAML and faster deployment, less stability, and **the need for Ops is not gone**.

## 11.9 Explore more

- Helm charts: package the Module 10 deployment so it installs with one command
- Kubeflow and Kubeflow Pipelines: training pipelines next to serving
- SageMaker and other managed serving: someone else runs the cluster
- KServe's V2 protocol (`/v2/models/<name>/infer`) and `InferenceGraph`